# 0. Page de garde

**Nom Prénom** : ALBURQUERQUE Julien  
**Formation** : CISIA — Concevoir et implémenter une solution d'intelligence artificielle (référentiel C1→C9)  
**Cas d'usage** : Détection précoce du décrochage étudiant en L1  
**Dépôt GitHub** : [cisia-decrochage-etudiant](https://github.com/jalb-code/cisia-decrochage-etudiant.git)  
**Date de soutenance** : 2026-09-01 — **Version du notebook** : v0.2 (2026-08-07)  
**Environnement** : Python 3.13 · `uv` (environnement figé par `uv.lock`) · scikit-learn — détail exhaustif en §15  

## Scaffolding du projet

- **Création de la structure du projet** — arborescence normalisée : code réutilisable dans `src/`, notebook unique, ADR dans `docs/adr/`, données hors dépôt.
- **Hygiène de code** : dépôt normalisé, format/lint (`ruff`), tests (`pytest`), `pre-commit` et intégration continue (GitHub Actions) → un livrable reproductible et vérifiable.

### Journal de bord

#### [ADR-0001](../docs/adr/0001-ne-pas-versionner-les-donnees.md) — Ne pas versionner les données
Les jeux de données sont des données scolaires **personnelles**. Même à faible volume, les inscrire dans Git les rendrait accessibles dans l'historique de façon quasi irréversible — y compris après suppression du fichier. Par **précaution RGPD**, je ne versionne rien : `data/` reste hors dépôt et seule l'arborescence (`raw` · `sample` · `gold`) est conservée via des `.gitkeep`. Contrepartie assumée : un clone n'est pas exécutable tel quel, l'approvisionnement des données est donc documenté dans `README.md` et `data/README.md`.


# 1. Résumé exécutif

## Problème

Dans une université pluridisciplinaire, une part élevée des étudiants de L1 abandonnent — **~28 % d'entre eux** (1 479 décrocheurs sur 5 200 étudiants de la cohorte). Aujourd'hui, l'abandon se constate en fin de semestre, quand il est trop tard pour agir.

## Objectif

Concevoir une solution d'IA capable de prédire le risque de décrochage à mi-S1, afin de proposer un accompagnement au bon moment.

**Deux prédictions :**
- **Risque d'abandon (Oui/Non)** — déterminer s'il faut mettre en place un accompagnement ;
- **Moyenne finale (note sur 20)** — calibrer et prioriser les accompagnements.

**Contraintes à prendre en compte :**
- Explicabilité de la prédiction exigée ;
- Usage éthique et conforme au RGPD des données.

## Approche

*TODO — à compléter*

## Résultats clés attendus

*TODO — à compléter après §12 : mettre en avant le rappel et l'AUC, plus la traduction métier (nb d'étudiants ciblés / détectés). Ne pas afficher l'accuracy.*

## Limites

*TODO — à compléter après §12 : cohorte unique et synthétique ; seuil à recalibrer par promotion.*

# 2. Cadrage métier et cas d'usage — journal de bord [C1]

## Problématique métier

Une université pluridisciplinaire constate un **taux d'abandon élevé en L1**. Aujourd'hui, l'abandon se **constate en fin de semestre** — trop tard pour agir.

La **direction de la réussite étudiante** veut identifier, **dès mi-parcours du S1**, les étudiants à risque de décrochage, pour déclencher un accompagnement (tutorat, soutien méthodologique, aide sociale).

> **Contrainte clé — agir tôt.** La détection se fait **à mi-S1**, avant que le décrochage ne soit consommé.

## Objectifs attendus

Deux prédictions complémentaires à mi-S1 :

| Cible | Variable | Tâche | Rôle |
|---|---|---|---|
| **Principale** | `abandon` (0/1) | Classification binaire | Prédire le risque de décrochage |
| **Secondaire** | `moyenne_finale` (/20) | Régression | Prioriser et calibrer l'intensité de l'accompagnement |

Trois exigences cadrent la solution dès le départ :

- **Aide à la décision, pas décision automatique.** La sortie est un **score probabiliste** ; l'équipe pédagogique reste **dans la boucle** et tranche l'accompagnement — la dimension conformité de ce choix est développée en **§4**.
- **Explicabilité — un critère de cadrage, pas un raffinement technique.** Les utilisateurs doivent comprendre **pourquoi** un étudiant est signalé, pour adapter l'accompagnement. Cette exigence est posée au cadrage et contraint la suite (choix du modèle, restitution).
- **Seuil de décision — non figé au cadrage.** Les modèles sont d'abord comparés sur des métriques **indépendantes du seuil** ; le choix du seuil, justifié par l'**asymétrie des coûts** (un faux négatif coûte plus cher qu'un faux positif), sera arbitré avec le métier — détaillé en **§12 [C8]**.

## Parties prenantes

| Acteur | Rôle | Attente / ce qu'on doit lui fournir |
|---|---|---|
| **Direction réussite étudiante** | Commanditaire | Une solution IA répondant aux objectifs qu'elle a définis. |
| **Tuteurs / responsables pédagogiques** | Utilisateurs | Un outil *user-friendly* qui, à mi-S1, signale le risque d'abandon, estime la moyenne finale et **explique** la prédiction. |
| **Étudiant** | Sujet des données **et bénéficiaire** de l'accompagnement | Minimisation des données et absence de biais (ex. `boursier`, `sexe`), avec une vigilance sur le risque de stigmatisation (développé en §4). |
| **DPO / référent éthique** | Garant de la conformité | Livrables de conformité à préciser (traités en §4). |
| **SI scolarité / LMS** | Fournisseur des données | Source des données (académiques, engagement LMS, administratives) ; nécessite un accès fiable et documenté ; l'intégration technique est précisée en §10-11 [C6/C7]. |

## Journal de bord

#### [ADR-0003](../docs/adr/0003-explicabilite-restitution-shap-local-global-thematisation.md) — Explicabilité : un critère de cadrage
Les utilisateurs (tuteurs, responsables pédagogiques) doivent comprendre **pourquoi** un étudiant est signalé pour adapter l'accompagnement. Nous inscrivons donc l'explicabilité **dès le cadrage** comme exigence, et non comme option technique tardive : elle contraint en aval le choix du modèle [C4] et la restitution [C8]. Le grain de restitution (par catégories métier, avec drill-down par feature) et la méthode (SHAP) sont tranchés dans l'ADR.


# 3. Données : disponibilité, gouvernance et alternatives — journal de bord [C1]

vérifier disponibilité/accès ;
intégrer le catalogue des formations ; envisager des alternatives en cas de données manquantes.

# 4. Enjeux éthiques, sociétaux et conformité — journal de bord [C2]

# 5. Chargement et compréhension des données [C3]

**Objectif** : établir ce que les fichiers contiennent, comment c'est écrit et ce qui cloche - puis conformer les données afin de produire le palier **bronze** sur lequel portera l'EDA (§6).

> 🔧 **Comment je m'y prends** - Le profilage est produit par un **script Python** (`profiling.profile_csv`) qui déduit tout du contenu de chaque fichier : encodage, délimiteur, puis, colonne par colonne, type, motifs d'écriture, bornes, manquants et non-conformité. Il en écrit **un rapport HTML complet par fichier** qui me sert de support à mes constatations et mes décisions.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from decrochage_l1 import schema
from decrochage_l1.config import settings
from decrochage_l1.data import bronze, cleaning, profiling

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 38)

# `report_dir` déclenche l'écriture du rapport HTML complet - hors dépôt, car il cite
# des valeurs brutes. Sans lui, mesurer ne touche pas au disque.
profils = {
    chemin.name: profiling.profile_csv(chemin, report_dir=settings.report_dir)
    for chemin in sorted(settings.raw_dir.glob("*.csv"))
}
profil_catalogue, profil_etudiants = sorted(profils.values(), key=lambda profil: profil.file.n_rows)

# Liens vers les rapports HTML complets
liens = "\n".join(
    f"- [{profil.report_path.name}]"
    f"(../{profil.report_path.relative_to(settings.root_dir).as_posix()})"
    for profil in profils.values()
    if profil.report_path is not None
)
display(Markdown(f"**> Rapports de profilage générés**\n\n{liens}"))

**> Rapports de profilage générés**

- [profil-dataset_catalogue_formations_V5.html](../reports/profil-dataset_catalogue_formations_V5.html)
- [profil-dataset_decrochage_etudiants_complet_V5.html](../reports/profil-dataset_decrochage_etudiants_complet_V5.html)

## 5.1 Profilage des fichiers

In [2]:
# Une ligne par fichier : encodage et délimiteur sont déduits du contenu.
display(
    pd.concat(
        [profil.file.to_frame().set_index("Propriété").T for profil in profils.values()],
        ignore_index=True,
    )
    .rename_axis(columns=None)
    .set_index("Fichier")
)

,Encodage,Délimiteur,Lignes,Colonnes,Lignes en double,Taille
Fichier,,,,,,
dataset catalogue_formations_V5.csv,utf-8-sig (avec BOM),"« , »",8,7,0,0.5 Kio
dataset decrochage_etudiants_complet_V5.csv,utf-8-sig (avec BOM),"« , »",5 240,33,40,887.4 Kio


**Constat ⇒ Décision**

- **40 paires strictement identiques** sur les 33 colonnes ⇒ Supprimer les lignes en double

## 5.2 Profilage des colonnes et de leur contenu

> 🔎 **Où lire le détail complet.** L'`overview` ci-dessous donne les huit indicateurs qui tiennent à l'écran. Les six autres - motifs d'écriture, non-conformité, et surtout l'**inventaire exhaustif des écritures rencontrées, modalité par modalité** - sont dans le **rapport HTML** généré en tête de §5, mon support de travail pour ces constats.

In [3]:
display(Markdown("**> Catalogue des formations**"))
display(profil_catalogue.overview())

**> Catalogue des formations**

,colonne,type_reel,type_semantique,n_distinct,n_distinct_normalise,min,max,null_%
0,filiere,str,texte,8,8,,,0.0
1,faculte,str,texte,6,6,,,0.0
2,niveau,str,constant,1,1,,,0.0
3,ects_semestre,int64,entier,2,2,28,30,0.0
4,capacite_accueil,int64,entier,8,8,480,980,0.0
5,volume_horaire_s1,int64,entier,7,7,247,325,0.0
6,taux_reussite_historique_pct,int64,entier,7,7,60,81,0.0


**Constat ⇒ Décision**

- `filiere` porte 8 valeurs pour 8 lignes ⇒ c'est la clé primaire de la table et sera utilisée comme clé de jointure.
- `niveau` est **constant** (`L1`), donc sans information ⇒ exclusion de principe sur le **gold dataset**

**⇒ Rien à mettre en forme.**

In [4]:
display(Markdown("**> Étudiants**"))
display(profil_etudiants.overview())

**> Étudiants**

,colonne,type_reel,type_semantique,n_distinct,n_distinct_normalise,min,max,null_%
0,student_id,str,identifiant,5200,5200,,,0.00
1,annee_universitaire,str,constant,1,1,,,0.00
2,filiere,str,categoriel,31,8,,,0.00
3,age,int64,entier,11,11,17,27,0.00
4,sexe,str,categoriel,11,8,,,0.00
5,bac_type,str,categoriel,12,7,,,0.00
6,mention_bac,str,categoriel,11,8,,,4.05
7,etablissement_origine,str,categoriel,4,4,,,4.98
8,boursier,str,booleen,8,6,,,0.00
9,distance_domicile_km,str,decimal,1233,1233,0.0,138.0,5.99


**Constat ⇒ Décision**
- `annee_universitaire` constante (`2024-2025`) ⇒ variance nulle, **à exclure** dans le **gold dataset**
- **Erreurs de format ⇒ à normaliser** - *la valeur est juste, l'écriture non*
  - Chiffrées restées en texte : `date_inscription` (3 formats), `taux_presence_pct` (« % » + virgule), `distance_domicile_km` (« km » + virgule), `moyenne_partiels_s1` (virgule)
  - Catégorielles (casse / accents / espaces) : `filiere` (31→8), `bac_type` (12→7), `sexe` (11→8), `mention_bac` (11→8), `boursier` (8→6)
- **Valeurs hétérogènes ⇒ à recoder** - *plusieurs libellés, une même valeur*
  - `sexe` : `f`/`femme` · `h`/`m`/`homme` · `nb`/`autre` · `nr`
  - `bac_type` : `gen`/`general`/`generale` · `techno`/`technologique` · `pro`/`professionnel`
  - `mention_bac` : `p`/`passable` · `ab`/`assez bien` · `b`/`bien` · `tb`/`tres bien`
  - `boursier` : `non`/`n`/`0` · `oui`/`o`/`1`

**Cohérence à vérifier ⇒ à faire en §5.3**
- `student_id`, `id_dossier` : un même identifiant porte-t-il deux lignes **divergentes**, hors des 40 doublons exacts ?
- `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` sur toutes les lignes ?
- `filiere` : **jointure** avec le catalogue réalisable ?

**Questions reportées ⇒ à traiter lors de l'EDA §6**
- Valeurs absentes - 10 colonnes, jusqu'à 49,29 % : mécanisme (refus, MAR…) à qualifier

## 5.3 Contrôles de cohérence

### 5.3.1 `student_id`, `id_dossier` : Identifiant unique hors des 40 lignes en double ?

In [5]:
etudiants = profil_etudiants.data
jumelles = etudiants.duplicated(keep=False)

for identifiant in ("student_id", "id_dossier"):
    conflits = int((etudiants[identifiant].duplicated(keep=False) & ~jumelles).sum())
    display(Markdown(f"- `{identifiant}` portant deux lignes divergentes : **{conflits}**"))

- `student_id` portant deux lignes divergentes : **0**

- `id_dossier` portant deux lignes divergentes : **0**

✅ **student_id** et **id_dossier** uniques dans le dataset - une fois les lignes en double supprimées, 1 ligne = 1 étudiant

### 5.3.2 `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` sur toutes les lignes ?

In [6]:
mesurees = sorted({colonne for paire in schema.ORDER_CONSTRAINTS for colonne in paire})
nombres = profil_etudiants.data[mesurees].apply(cleaning.parse_number)

display(profiling.check_order_constraints(nombres, schema.ORDER_CONSTRAINTS))

,contrainte,n_comparables,n_non_comparables,n_violations
0,nb_devoirs_rendus <= nb_devoirs_total,5240,0,0
1,nb_ue_validees_s1 <= nb_ue_total,5240,0,0


✅ `nb_devoirs_rendus ≤ nb_devoirs_total` et `nb_ue_validees_s1 ≤ nb_ue_total` toujours vrai

### 5.3.3 `filiere` : **jointure** avec le catalogue réalisable ?

In [7]:
communes = sorted(set(profil_etudiants.data.columns) & set(profil_catalogue.data.columns))
cle = communes[0]

# Rapprochement sur les valeurs NORMALISÉES : « Gestion » et « GESTION » sont la même
# filière. Comparées telles qu'écrites, elles compteraient toutes deux pour orphelines.
etudiants_cle = cleaning.normalize_text(profil_etudiants.data[cle])
catalogue_cle = cleaning.normalize_text(profil_catalogue.data[cle])

reference = set(catalogue_cle)
appariables = etudiants_cle.isin(reference)
orphelines = sorted(set(etudiants_cle.dropna()) - reference)
taux = 100 * appariables.mean()

with pd.option_context("display.max_colwidth", 100):
    display(
        pd.DataFrame(
            {
                "Mesure": [
                    "Nb filière (Étudiants)",
                    "Nb filière (Catalogue)",
                    "Lignes appariables",
                    "Filières hors catalogue",
                ],
                "Valeur": [
                    str(etudiants_cle.nunique()),
                    str(catalogue_cle.nunique()),
                    f"{appariables.sum()} / {len(etudiants_cle)} ({taux:.1f} %)",
                    ", ".join(orphelines) if orphelines else "aucune",
                ],
            }
        ).set_index("Mesure")
    )

,Valeur
Mesure,
Nb filière (Étudiants),8
Nb filière (Catalogue),8
Lignes appariables,5240 / 5240 (100.0 %)
Filières hors catalogue,aucune


✅ 100 % d'appariement ⇒ La jointure est possible

## 5.4 Thématisation

Je procède à une classification par thème des colonnes afin de rendre leur lecture plus aisée dans le cas où je mettrai en place la thématisation dans l'explicabilité des features.

In [8]:
colonnes = profil_etudiants.data.columns

with pd.option_context("display.max_colwidth", 90):
    display(
        pd.DataFrame(
            [{"colonnes": ", ".join(membres)} for membres in schema.COLUMN_THEMES.values()],
            index=pd.Index(schema.COLUMN_THEMES, name="thème"),
        )
    )

display(
    Markdown(
        f"Classées : **{len(schema.theme_by_column())} / {len(colonnes)}** · "
        f"non classées : **{schema.unclassified(colonnes) or 'aucune'}** · "
        f"citées mais absentes : **{schema.unknown(colonnes) or 'aucune'}**"
    )
)

,colonnes
thème,
Identifiants,"student_id, id_dossier"
Contexte d'inscription,"annee_universitaire, filiere, date_inscription"
Profil social et démographique,"age, sexe, boursier, distance_domicile_km, heures_travail_remunere_sem"
Parcours antérieur,"bac_type, mention_bac, etablissement_origine"
Engagement LMS,"connexions_lms_30j, heures_lms_total, ressources_consultees, messages_forum"
Assiduité et travail rendu,"taux_presence_pct, retards_rendus, nb_devoirs_total, nb_devoirs_rendus"
Résultats académiques,"moyenne_partiels_s1, nb_ue_total, nb_ue_validees_s1"
Ressenti déclaré,"motivation, satisfaction, sentiment_appartenance"
Avis du tuteur,commentaire_tuteur


Classées : **33 / 33** · non classées : **aucune** · citées mais absentes : **aucune**

## 5.5 Production du palier bronze

Le fichier écrit dans `data/bronze/` est une **copie de contrôle**, inspectable dans un tableur.
Ce n'est pas ce que §6 consomme : les `DataFrame` rendus ci-dessous circulent d'une section à l'autre, avec les types que la conformation leur a donnés.

In [9]:
jeux, rapports = {}, []

for nom, fichier in bronze.SOURCES.items():
    profil_source = profiling.profile_csv(settings.raw_dir / fichier)
    jeux[nom], rapport = bronze.build(profil_source, settings.bronze_dir / f"{nom}.csv")
    rapports.append(rapport)

etudiants_bronze, catalogue_bronze = jeux["etudiants"], jeux["catalogue"]

display(bronze.summary(rapports))

,fichier,lignes_source,doublons_retires,lignes_bronze,colonnes,colonnes_recodees
0,etudiants.csv,5240,40,5200,33,4
1,catalogue.csv,8,0,8,7,0


✅ Palier **Bronze** construit

# 6. Analyse exploratoire (EDA) : visualisations et interprétation — journal de bord [C3]


# 7. Préparation des données (nettoyage, manquants, transformations, features) — journal de bord [C3]

# 8. Choix du modèle et démarche scientifique (baseline, modèles, comparaison) — journal de bord [C4]

# 9. Entraînement, validation et ajustement (sélection du modèle final) — journal de bord [C5]

# 10. Implémentation et mise en exploitation (déploiement, exemple d’usage) — journal de bord [C6]

# 11. Architecture cible et contraintes [C7]

# 12. Mesure de performance et impacts (métriques techniques + métier) [C8]

# 13. Amélioration continue (ré-entraînement, suivi, versioning) [C9]

# 14. Conclusion (synthèse et recommandations)


# 15. Annexes (versions, paramètres, dépendances, fonctions utilitaires)